# Day 24 · 评测报告 v1

**配套讲义**: [`days/day-24.md`](../days/day-24.md) ｜ **本地可跑，不需要 GPU**

把通用集、领域集、幻觉集的所有结果合成一份 `reports/eval_v1.md`，**每个结论都能追溯到数据**，没有一句「感觉」。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w4.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 生成报告

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.eval.report",
                    "--runs", "reports/eval_base_raw.jsonl", "reports/eval_lora_raw.jsonl",
                    "--out", "reports/eval_v1.md"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout[-2000:] or r.stderr[-2000:])

## 2. 逐句检查「有没有感觉词」

自动扫一遍报告里的主观词汇 —— 这是最有效的自查方式。

In [ ]:
import re
from pathlib import Path

p = Path("../reports/eval_v1.md")
FUZZY = ["感觉", "似乎", "大概", "可能好一些", "明显更好", "应该", "差不多"]
if p.exists():
    text = p.read_text()
    hits = []
    for i, line in enumerate(text.splitlines(), 1):
        for w in FUZZY:
            if w in line:
                hits.append((i, w, line.strip()[:80]))
    if hits:
        print(f"⚠️  发现 {len(hits)} 处主观表述：")
        for i, w, line in hits:
            print(f"  L{i} [{w}] {line}")
        print("\n→ 逐条改成数据表述，例如把「L4 明显更差」写成「L4 规则命中 43.8% vs 基座 47.9%，Δ-4.1」")
    else:
        print("✓ 没有明显的主观表述")
else:
    print("先跑上面的生成格子")

## 3. M4 验收自问

- 如果只能改一件事提升模型，你会改什么？**依据是哪一行数据？**
- 这份报告里，哪一个结论是你最没把握的？为什么？

In [ ]:
self_check = """
最该改的一件事：
依据的数据：
最没把握的结论：
"""
print(self_check)

## 验收清单

- [ ] `reports/eval_v1.md` 含：通用集分数 + 领域集分层分 + 幻觉率对照 + 失败模式 + 下一步
- [ ] `make eval-fake` 能区分好坏：**好 > 坏 > 全拒答**，且三个数不接近
- [ ] 报告里**每一个结论都有数字支撑**，没有「感觉」「似乎」
- [ ] 报告里写明了模型版本、数据版本、评测集版本（可复现）
- [ ] `progress/weekly-review.md` 的 W4 段已写；进度表 W4 六天 `[x]`，M4 打卡

**卡住了？** 回看 [`days/day-24.md`](../days/day-24.md) 第五节「容易踩的坑」。

> **明天**：`days/day-25.md` —— W5 对齐周：DPO 原理